# Wan 2.2 Генератор Видео — Colab T4

**Ноутбук «всё в одном»** для генерации видео с ComfyUI + Wan 2.2 14B на бесплатном T4 GPU.

**Время настройки:** ~10-15 мин (скачивание моделей)
**VRAM:** ~14 GB пиковое потребление

---

### Как это работает (пошагово)

```
[1. GPU]  →  [2. Установка]  →  [3. Модели]  →  [4. Воркфлоу]  →  [5. ЗАГРУЗКА МЕДИА]  →  [6. Запуск]
                                                                          ↑
                                                                 Самый важный шаг!
                                                         Загрузите нужные файлы для
                                                         выбранного воркфлоу (см. ниже)
```

---

### Какой воркфлоу выбрать?

| Воркфлоу | Задача | Вход | Выход | Время на T4 |
|----------|--------|------|-------|-------------|
| `video_wan_long_ultimate` | Длинное видео из картинки | Изображение + текст | Видео 20-30с (I2V + VACE) | ~15-25 мин |
| `video_wan_t2v` | Видео из текста | Только текст | Видео 3.4с | ~3-5 мин |
| `video_wan_clip` | Быстрый клип для монтажа | Изображение + текст | Клип 3.4с | ~3-5 мин |
| `video_wan_v2v` | Рестайлинг видео | Видео + текст | Видео (той же длины) | ~5-10 мин |
| `video_wan_talking` | Говорящая голова | Фото + аудио | Видео с lip-sync | ~5-10 мин |
| `video_wan_reels` | Reels из набора фото | Папки с фото | Серия клипов | ~10-15 мин |

---

### Что загружать? (ячейка 5)

| Режим | Что нужно загрузить | Форматы | Рекомендации |
|-------|---------------------|---------|-------------|
| **I2V** (long, clip) | 1 изображение | .jpg, .png | Чёткое фото, хорошее освещение |
| **Talking** | 1 портретное фото + 1 аудио | Фото: .jpg, .png / Аудио: .wav, .mp3 | Фото анфас, чистый аудиотрек |
| **V2V** | 1 видео | .mp4 | Короткое видео, стабильная камера |
| **T2V** | Ничего не нужно | — | Только текстовый промпт в ComfyUI |
| **Reels** | Набор изображений | .jpg, .png | Загрузите и настройте пути в воркфлоу |

**Запускайте ячейки 1-6 по порядку**, затем откройте ссылку на туннель.

In [ ]:
#@title 1. Проверка GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} ГБ")

In [ ]:
#@title 2. Установка ComfyUI + Кастомные ноды
%cd /content
!test -d ComfyUI || git clone https://github.com/comfyanonymous/ComfyUI.git
%cd /content/ComfyUI

# Фиксируем numpy < 2.0
!echo "numpy<2.0" > /tmp/numpy_constraint.txt
!pip install "numpy>=1.26,<2.0" -q
!pip install -r requirements.txt -q -c /tmp/numpy_constraint.txt

# Кастомные ноды
%cd /content/ComfyUI/custom_nodes
!test -d ComfyUI-WanVideoWrapper || git clone https://github.com/kijai/ComfyUI-WanVideoWrapper.git
!test -d ComfyUI-VideoHelperSuite || git clone https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git
!test -d ComfyUI-KJNodes || git clone https://github.com/kijai/ComfyUI-KJNodes.git

!pip install -r ComfyUI-WanVideoWrapper/requirements.txt -q -c /tmp/numpy_constraint.txt
!pip install -r ComfyUI-VideoHelperSuite/requirements.txt -q -c /tmp/numpy_constraint.txt
!pip install -r ComfyUI-KJNodes/requirements.txt -q -c /tmp/numpy_constraint.txt

print("\nУстановлено!")

In [ ]:
#@title 3. Скачивание моделей (~28 ГБ, 5-10 мин)
import os

M = "/content/ComfyUI/models"
os.makedirs(f"{M}/diffusion_models", exist_ok=True)
os.makedirs(f"{M}/text_encoders", exist_ok=True)
os.makedirs(f"{M}/vae", exist_ok=True)

HF = "https://huggingface.co/Kijai/WanVideo_comfy/resolve/main"

files = {
    # Wan 2.2 T2V 14B fp8 (~14 ГБ)
    f"{M}/diffusion_models/Wan2_2-T2V-A14B-LOW_fp8_e4m3fn_scaled_KJ.safetensors":
        f"{HF}/Wan2_2-T2V-A14B-LOW_fp8_e4m3fn_scaled_KJ.safetensors",
    # Модуль VACE 14B (~5.7 ГБ)
    f"{M}/diffusion_models/Wan2_2_Fun_VACE_module_A14B_LOW_bf16.safetensors":
        f"{HF}/Wan2_2_Fun_VACE_module_A14B_LOW_bf16.safetensors",
    # FantasyTalking (~0.5 ГБ)
    f"{M}/diffusion_models/fantasytalking_fp16.safetensors":
        f"{HF}/fantasytalking_fp16.safetensors",
    # Текстовый энкодер UMT5 XXL fp8 (~6.3 ГБ)
    f"{M}/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors":
        f"{HF}/umt5_xxl_fp8_e4m3fn_scaled.safetensors",
    # VAE (~1.3 ГБ)
    f"{M}/vae/wan2.2_vae.safetensors":
        f"{HF}/wan2.2_vae.safetensors",
}

for dest, url in files.items():
    if os.path.exists(dest):
        print(f"Уже существует: {os.path.basename(dest)}")
    else:
        print(f"\nСкачивание {os.path.basename(dest)}...")
        !wget -q --show-progress -O "{dest}" "{url}"

# Валидация скачивания
for dest in files:
    if os.path.exists(dest) and os.path.getsize(dest) < 1024:
        os.remove(dest)
        print(f"  ОШИБКА: {os.path.basename(dest)} не скачан — перезапустите ячейку")

print("\nВсе модели готовы!")
!du -sh {M}/diffusion_models/* {M}/text_encoders/* {M}/vae/*

In [ ]:
#@title 4. Скачивание воркфлоу
import os

GIST = "ea1ab4a53e1be1b8401e5031bcdae4f3"
RAW = f"https://gist.githubusercontent.com/russiankendricklamar/{GIST}/raw"
WF_DIR = "/content/ComfyUI/user/default/workflows"
os.makedirs(WF_DIR, exist_ok=True)

workflows = [
    "video_wan_long_ultimate.json",
    "video_wan_t2v.json",
    "video_wan_clip.json",
    "video_wan_v2v.json",
    "video_wan_talking.json",
]

# Воркфлоу для Colab (фиксированные пути)
colab_workflows = {
    "video_wan_reels.json": "video_wan_reels_colab.json",
}

for wf in workflows:
    !wget -q -O "{WF_DIR}/{wf}" "{RAW}/{wf}"
    print(f"Скачано: {wf}")

for local_name, gist_name in colab_workflows.items():
    !wget -q -O "{WF_DIR}/{local_name}" "{RAW}/{gist_name}"
    print(f"Скачано: {gist_name} -> {local_name}")

print(f"\n{len(workflows) + len(colab_workflows)} воркфлоу сохранено в {WF_DIR}")

In [ ]:
#@title 5. Загрузка медиа (фото / аудио / видео)
#@markdown ## Нажмите кнопку "Выбрать файлы" ниже
#@markdown
#@markdown Загрузите файлы, необходимые для выбранного воркфлоу.
#@markdown Файлы сохранятся в папку `/content/ComfyUI/input/` — именно оттуда
#@markdown ComfyUI будет брать данные при генерации.
#@markdown
#@markdown ---
#@markdown
#@markdown ### Что загружать для каждого воркфлоу:
#@markdown
#@markdown | Воркфлоу | Что загрузить | Форматы |
#@markdown |----------|---------------|---------|
#@markdown | **long_ultimate / clip** | 1 изображение | .jpg, .png |
#@markdown | **talking** | 1 портретное фото + 1 аудио | .jpg/.png + .wav/.mp3 |
#@markdown | **v2v** | 1 видео | .mp4 |
#@markdown | **t2v** | Ничего (пропустите эту ячейку) | — |
#@markdown | **reels** | Набор изображений | .jpg, .png |
#@markdown
#@markdown ---
#@markdown
#@markdown ### Требования к файлам:
#@markdown - Изображения: **чёткие**, хорошее освещение, без водяных знаков
#@markdown - Аудио (talking): **чистая речь**, без фонового шума, 5-30 секунд
#@markdown - Видео (v2v): **короткое** (3-10с), стабильная камера

from google.colab import files
from IPython.display import display, Image as IPImage
import os

INPUT_DIR = "/content/ComfyUI/input"
os.makedirs(INPUT_DIR, exist_ok=True)

VALID_IMAGES = ('.jpg', '.jpeg', '.png')
VALID_AUDIO = ('.wav', '.mp3')
VALID_VIDEO = ('.mp4',)
VALID_ALL = VALID_IMAGES + VALID_AUDIO + VALID_VIDEO

print("=" * 60)
print("  ЗАГРУЗКА МЕДИА ДЛЯ ГЕНЕРАЦИИ ВИДЕО")
print("=" * 60)
print(f"\n  Папка: {INPUT_DIR}")
print("  Нажмите кнопку ниже и выберите нужные файлы\n")

uploaded = files.upload()

images_saved = []
audio_saved = []
video_saved = []
skipped = []

for name, data in uploaded.items():
    ext = os.path.splitext(name)[1].lower()
    if ext not in VALID_ALL:
        skipped.append(name)
        continue

    path = os.path.join(INPUT_DIR, name)
    with open(path, "wb") as f:
        f.write(data)

    if ext in VALID_IMAGES:
        images_saved.append(name)
    elif ext in VALID_AUDIO:
        audio_saved.append(name)
    elif ext in VALID_VIDEO:
        video_saved.append(name)

# Итоговая сводка
print(f"\n{'=' * 60}")
print("  РЕЗУЛЬТАТ ЗАГРУЗКИ")
print(f"{'=' * 60}")

if images_saved:
    print(f"\n  Изображения ({len(images_saved)}):")
    for f in images_saved:
        print(f"    {f}")

if audio_saved:
    print(f"\n  Аудио ({len(audio_saved)}):")
    for f in audio_saved:
        print(f"    {f}")

if video_saved:
    print(f"\n  Видео ({len(video_saved)}):")
    for f in video_saved:
        print(f"    {f}")

if skipped:
    print(f"\n  Пропущено (неподдерживаемый формат: {len(skipped)}):")
    for f in skipped:
        print(f"    {f}")
    print("  Поддерживаемые форматы: .jpg, .png, .wav, .mp3, .mp4")

total = len(images_saved) + len(audio_saved) + len(video_saved)
print(f"\n  Всего загружено: {total} файлов")
print(f"  Папка: {INPUT_DIR}")
print(f"{'=' * 60}")

# Подсказка по готовности
if images_saved and audio_saved:
    print("\n  Готово для: long_ultimate, clip, talking")
elif images_saved:
    print("\n  Готово для: long_ultimate, clip, reels")
    print("  Для talking — загрузите ещё аудиофайл (.wav/.mp3)")
elif audio_saved:
    print("\n  Для talking — загрузите ещё портретное фото (.jpg/.png)")
elif video_saved:
    print("\n  Готово для: v2v")

# Превью изображений
if images_saved:
    print(f"\nПревью загруженных изображений:")
    for f in images_saved[:6]:
        path = os.path.join(INPUT_DIR, f)
        print(f"  {f}")
        try:
            display(IPImage(filename=path, width=300))
        except Exception:
            pass

In [ ]:
#@title 6. Запуск ComfyUI
#@markdown ---
#@markdown ### Способ подключения
#@markdown Cloudflare рекомендуется. Если страница белая — переключите на **localtunnel**.
tunnel_method = "cloudflare" #@param ["cloudflare", "localtunnel"]

import subprocess, time, re, os, requests

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
LOG_FILE = "/content/comfyui.log"

comfy = subprocess.Popen(
    ["python", "main.py", "--listen", "0.0.0.0", "--port", "8188",
     "--enable-cors-header", "*"],
    cwd="/content/ComfyUI",
    stdout=open(LOG_FILE, "w"),
    stderr=subprocess.STDOUT
)

print("Запуск ComfyUI...")
started = False
for i in range(24):
    time.sleep(5)
    try:
        if requests.get("http://localhost:8188/api/system_stats", timeout=3).status_code == 200:
            print(f"  ComfyUI запущен за {(i+1)*5} сек!")
            started = True
            break
    except Exception:
        pass

if not started:
    print("  ComfyUI не ответил за 120 сек.")
    with open(LOG_FILE) as f:
        lines = f.readlines()
        errors = [l for l in lines if any(w in l.lower() for w in ['error', 'traceback', 'exception', 'failed'])]
        if errors:
            print("  --- Ошибки ---")
            for l in errors[-15:]:
                print(f"  {l.rstrip()}")
        for l in lines[-10:]:
            print(f"  {l.rstrip()}")
    raise RuntimeError("ComfyUI не запустился. Проверьте ошибки выше.")

with open(LOG_FILE) as f:
    log_text = f.read()
import_errors = [l for l in log_text.split('\n') if 'cannot import' in l.lower() or 'no module named' in l.lower()]
if import_errors:
    print(f"\n  Предупреждения при загрузке нод ({len(import_errors)}):")
    for l in import_errors[:5]:
        print(f"    {l.strip()}")

url = None
if tunnel_method == "cloudflare":
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
    !dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
    tunnel = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://localhost:8188"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
    time.sleep(5)
    for _ in range(20):
        line = tunnel.stdout.readline().decode()
        match = re.search(r"https://[\w-]+\.trycloudflare\.com", line)
        if match:
            url = match.group(0)
            break
else:
    !npm install -g localtunnel > /dev/null 2>&1
    tunnel = subprocess.Popen(
        ["lt", "--port", "8188"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
    time.sleep(8)
    for _ in range(20):
        line = tunnel.stdout.readline().decode()
        match = re.search(r"https://[\w-]+\.loca\.lt", line)
        if match:
            url = match.group(0)
            break

if url:
    print(f"\n{'='*60}")
    print(f"  ComfyUI готов! Откройте: {url}")
    print(f"{'='*60}")
    if tunnel_method == "localtunnel":
        print(f"\n  Localtunnel попросит пароль — нажмите ссылку на странице.")
    print(f"\n  Загрузка воркфлоу: Меню -> Load -> выберите любой video_wan_* воркфлоу")
    print(f"\n  Если страница белая:")
    print(f"    1. Обновите страницу (Ctrl+R)")
    alt = "localtunnel" if tunnel_method == "cloudflare" else "cloudflare"
    print(f"    2. Если не помогло — перезапустите с tunnel_method = '{alt}'")
    print(f"\n  Логи ComfyUI: !cat {LOG_FILE}")
else:
    alt = "localtunnel" if tunnel_method == "cloudflare" else "cloudflare"
    print(f"\nURL туннеля не найден. Попробуйте: tunnel_method = '{alt}'")

In [ ]:
#@title 7. Сохранение на Google Drive
from google.colab import drive
import shutil, glob, os

drive.mount('/content/drive')

src = "/content/ComfyUI/output"
dst = "/content/drive/MyDrive/ComfyUI_Output/wan"
os.makedirs(dst, exist_ok=True)

videos = glob.glob(f"{src}/*.mp4") + glob.glob(f"{src}/*.png")
if not videos:
    print("Результатов пока нет. Сначала сгенерируйте видео!")
else:
    for v in videos:
        shutil.copy2(v, dst)
        print(f"Скопировано: {os.path.basename(v)}")
    print(f"\n{len(videos)} файлов сохранено в {dst}")

---
## Гайд по воркфлоу

---

### `video_wan_long_ultimate` — Длинное I2V (20-30с)

```
Изображение + Промпт  →  [VACE контекстные окна]  →  Длинное связное видео 20-30с
```

Загрузите изображение, задайте промпт, нажмите Queue. VACE использует контекстные окна для генерации длинного связного видео без потери качества.

| Параметр | Значение | Описание |
|----------|----------|----------|
| `num_frames` | 481 / 721 | 481 = 20с, 721 = 30с |
| `blocks_to_swap` | 20 | По умолчанию для T4. Увеличьте до 30 при OOM |

---

### `video_wan_t2v` — Текст-в-видео

```
Только промпт  →  [Wan 2.2 14B T2V]  →  Видео 3.4с
```

Чистая генерация видео по текстовому описанию. Изображение не нужно.

| Параметр | Значение | Описание |
|----------|----------|----------|
| `num_frames` | 81 | По умолчанию 3.4с. Увеличьте для более длинных видео |

---

### `video_wan_clip` — Короткий клип (3.4с)

```
Изображение (опционально) + Промпт  →  [Wan 2.2 I2V]  →  Клип 3.4с
```

Быстрая генерация короткого клипа из стартового изображения. Идеален для создания нескольких клипов под монтаж.

---

### `video_wan_v2v` — Видео-в-видео

```
Референсное видео + Промпт  →  [VACE рестайлинг]  →  Видео в новом стиле
```

Загрузите видео, опишите желаемый стиль в промпте. VACE сохраняет движение и структуру оригинала, меняя визуальный стиль.

---

### `video_wan_talking` — Говорящая голова

```
Портретное фото + Аудиофайл  →  [FantasyTalking]  →  Видео с lip-sync
```

Загрузите портретное фото и аудиофайл. Генерирует видео говорящей головы с синхронизацией губ.

| Параметр | Значение | Описание |
|----------|----------|----------|
| `audio_scale` | по умолчанию | Сила синхронизации губ |

Используйте ячейку 8 (TTS) ниже для генерации речи, если нет готового аудио.

---

### `video_wan_reels` — Reels из папок

```
Папки с изображениями  →  [Wan 2.2 I2V пакетно]  →  Серия клипов
```

Загружает изображения из папок и генерирует серию клипов. На Colab загрузите изображения через ячейку 5 и настройте пути к папкам в воркфлоу.

---

### Пресеты разрешений

| Размер | Соотношение | Применение |
|--------|-------------|------------|
| 832x480 | 16:9 | Альбомная ориентация (YouTube) |
| 480x832 | 9:16 | Портрет / Reels / Shorts |
| 608x1080 | 9:16 | HD Портрет (больше деталей) |

---
## Дополнительные инструменты
Запускайте эти ячейки по необходимости — они не зависят от основной настройки.

In [ ]:
#@title 8. Генерация речи (TTS)
#@markdown Генерация аудио для воркфлоу **talking** с помощью edge-tts.

text = "Hello, this is a demo of the talking head generator." #@param {type:"string"}
voice = "en-US-AriaNeural" #@param ["en-US-AriaNeural", "en-US-GuyNeural", "ru-RU-SvetlanaNeural", "ru-RU-DmitryNeural"]
output_name = "speech.wav" #@param {type:"string"}

!pip install edge-tts -q 2>/dev/null
!edge-tts --voice "{voice}" --text "{text}" --write-media /content/ComfyUI/input/{output_name}

# Предпрослушивание
from IPython.display import Audio, display
display(Audio(f"/content/ComfyUI/input/{output_name}"))
print(f"\nСохранено: /content/ComfyUI/input/{output_name}")
print("Выберите этот файл в ноде LoadAudio в ComfyUI.")

In [ ]:
#@title 9. Улучшение промпта с Qwen2-VL
#@markdown Описание загруженного изображения с помощью Qwen2-VL 2B для генерации видео-промпта.
#@markdown Работает на CPU — занимает 1-2 мин.

image_name = "my_image.png" #@param {type:"string"}

!pip install transformers qwen-vl-utils -q 2>/dev/null

import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

print("Загрузка Qwen2-VL 2B на CPU...")
model = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct",
    torch_dtype=torch.float16,
    device_map="cpu"
)
processor = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-2B-Instruct")

image_path = f"/content/ComfyUI/input/{image_name}"
messages = [{
    "role": "user",
    "content": [
        {"type": "image", "image": f"file://{image_path}"},
        {"type": "text", "text": (
            "Describe this image in detail for use as a video generation prompt. "
            "Include: subject, action/pose, setting, lighting, colors, mood, "
            "camera angle. Write as one continuous paragraph, no bullet points."
        )}
    ]
}]

text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt")

print("Генерация промпта...")
with torch.no_grad():
    output_ids = model.generate(**inputs, max_new_tokens=256)

prompt = processor.batch_decode(
    output_ids[:, inputs.input_ids.shape[1]:],
    skip_special_tokens=True
)[0]

print(f"\n{'='*60}")
print("СГЕНЕРИРОВАННЫЙ ПРОМПТ (скопируйте в ComfyUI):")
print(f"{'='*60}")
print(prompt)

# Освобождение памяти
del model, processor
import gc; gc.collect()

In [ ]:
#@title 10. Монтаж: объединение видео
#@markdown Склеивание всех сгенерированных видео из output ComfyUI в один файл.

import glob, os

output_dir = "/content/ComfyUI/output"
videos = sorted(glob.glob(f"{output_dir}/*.mp4"))

if not videos:
    print("Видео не найдены в output/. Сначала сгенерируйте видео!")
else:
    print(f"Найдено {len(videos)} видео:")
    for i, v in enumerate(videos):
        print(f"  {i+1}. {os.path.basename(v)}")

    # Запись списка файлов для ffmpeg
    with open("/content/filelist.txt", "w") as f:
        for v in videos:
            f.write(f"file '{v}'\n")

    # Склеивание
    !ffmpeg -y -f concat -safe 0 -i /content/filelist.txt -c copy /content/montage.mp4 2>/dev/null

    print(f"\nМонтаж сохранён: /content/montage.mp4")
    !ls -lh /content/montage.mp4

    # Скачивание
    from google.colab import files
    files.download("/content/montage.mp4")

---
## Скоро: Пайплайн AI-музыкальных клипов

**Концепция:** Загрузите музыкальный трек и автоматически сгенерируйте синхронизированный музыкальный клип.

### Как это будет работать
1. **Детекция битов** (librosa) — анализ трека, поиск битов, сегментов, уровней энергии
2. **Генерация промптов** — создание видео-промпта для каждого сегмента на основе текста/настроения
3. **Генерация клипов** (ComfyUI API) — генерация короткого видеоклипа на каждый бит-сегмент через `video_wan_clip`
4. **Сборка** (FFmpeg) — склейка клипов, синхронизация с аудио, добавление переходов

### Ориентировочное время
- ~45-90 мин для 3-минутного видео на T4
- ~15-20 клипов по 3.4с каждый

### Стек
- `librosa` — детекция битов/onset, анализ темпа
- ComfyUI REST API — постановка в очередь video_wan_clip по сегментам
- `ffmpeg` — склейка + синхронизация аудио

**Это будет отдельный ноутбук: `colab_music_video.ipynb`**